In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif
import numpy as np
import os
import warnings
import pywt
from scipy import signal
from scipy.stats import skew, kurtosis
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

warnings.filterwarnings("ignore")

focal_folder_path = '/Users/gaurav/Downloads/Complete set/Data_F_Ind_1_750'
non_focal_folder_path = '/Users/gaurav/Downloads/Complete set/Data_N_Ind_1_750'

def low_pass_filter(data, cutoff=250, fs=512, order=5):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = signal.butter(order, normal_cutoff, btype='low', analog=False)
    filtered_data = signal.filtfilt(b, a, data)
    return filtered_data

def sevcik_fractal_dimension(data):
    L = np.sum(np.sqrt(1 + np.diff(data)**2))
    return L / len(data)

def wavelet_features(data, wavelet='db4', level=4):
    coeffs = pywt.wavedec(data, wavelet, level=level)
    features = [np.mean(c) for c in coeffs] + [np.std(c) for c in coeffs]
    return features

def hurst_exponent(data):
    N = len(data)
    T = np.arange(1, N + 1)
    Y = np.cumsum(data - np.mean(data))
    R = np.max(Y) - np.min(Y)
    S = np.std(data)
    return R / S if S != 0 else 0

def higuchi_fractal_dimension(data, kmax=10):
    Lk = []
    N = len(data)
    for k in range(1, kmax + 1):
        Lmk = []
        for m in range(k):
            Lmki = 0
            for i in range(1, int((N - m) / k)):
                Lmki += abs(data[m + i * k] - data[m + (i - 1) * k])
            norm_factor = (N - 1) / (int((N - m) / k) * k)
            Lmk.append(Lmki * norm_factor)
        Lk.append(np.mean(Lmk))
    hfd = np.polyfit(np.log(range(1, kmax + 1)), np.log(Lk), 1)[0]
    return hfd

def katz_fractal_dimension(data):
    d = np.max(np.sqrt((np.arange(len(data)) - 0)**2 + (data - data[0])**2))
    L = np.sum(np.sqrt(1 + np.diff(data)**2))
    return np.log10(L) / (np.log10(d) + np.log10(len(data)))

def power_spectral_density(data):
    freqs, psd = signal.welch(data, fs=512)
    return np.mean(psd)

def spectral_entropy(data):
    power_spectrum = np.abs(np.fft.fft(data))**2
    power_spectrum /= np.sum(power_spectrum)
    return -np.sum(power_spectrum * np.log2(power_spectrum + 1e-10))

def bandpower(data, sf, band, window_sec=None):
    band = np.asarray(band)
    low, high = band
    if window_sec:
        nperseg = int(window_sec * sf)
    else:
        nperseg = None
    freqs, psd = signal.welch(data, sf, nperseg=nperseg)
    freq_res = freqs[1] - freqs[0]
    idx_band = np.logical_and(freqs >= low, freqs <= high)
    bp = np.trapz(psd[idx_band], dx=freq_res)
    return bp

def svd_features(data):
    u, s, vh = np.linalg.svd(data.reshape(-1, 1), full_matrices=False)
    return np.mean(s), np.std(s)

def dmd_features(data, rank=5):
    x = data[:-1].reshape(-1, 1)
    y = data[1:].reshape(-1, 1)
    u, s, vh = np.linalg.svd(x, full_matrices=False)
    s = np.diag(s)
    u_r = u[:, :rank]
    s_r = s[:rank, :rank]
    vh_r = vh[:rank, :]
    A_tilde = u_r.T @ y @ vh_r.T @ np.linalg.inv(s_r)
    eigenvalues = np.linalg.eigvals(A_tilde)
    return np.real(eigenvalues).mean(), np.imag(eigenvalues).mean()
def load_data_with_features(folder_path, limit=3750):
    file_list = [f for f in os.listdir(folder_path) if f.endswith('.txt')][:limit]
    data_array = []
    for file_name in file_list:
        file_path = os.path.join(folder_path, file_name)
        data = np.loadtxt(file_path, delimiter=',').flatten()
        
        filtered_data = low_pass_filter(data)
        
        hfd = higuchi_fractal_dimension(filtered_data) 
        kfd = katz_fractal_dimension(filtered_data)
        psd = power_spectral_density(filtered_data)
        se = spectral_entropy(filtered_data)
        sevcik = sevcik_fractal_dimension(filtered_data)
        hurst = hurst_exponent(filtered_data)
        wavelet_feat = wavelet_features(filtered_data)
        rms_val = np.sqrt(np.mean(filtered_data**2))
        zcr_val = ((filtered_data[:-1] * filtered_data[1:]) < 0).sum() / len(filtered_data)
        mean_svd, std_svd = svd_features(filtered_data)
        real_dmd, imag_dmd = dmd_features(filtered_data)
        sf = 250
        delta_bp = bandpower(filtered_data, sf, [0.5, 4])
        theta_bp = bandpower(filtered_data, sf, [4, 8])
        alpha_bp = bandpower(filtered_data, sf, [8, 12])
        beta_bp = bandpower(filtered_data, sf, [12, 30])
        gamma_bp = bandpower(filtered_data, sf, [30, 100])
        
        features = [
            hfd, kfd, psd, se, sevcik, hurst,
            delta_bp, theta_bp, alpha_bp, beta_bp, gamma_bp,
            rms_val, zcr_val, mean_svd, std_svd, real_dmd, imag_dmd
        ] + wavelet_feat
        data_array.append(features)
    return np.array(data_array)

focal_data = load_data_with_features(focal_folder_path)
non_focal_data = load_data_with_features(non_focal_folder_path)

focal_labels = np.ones(len(focal_data))
non_focal_labels = np.zeros(len(non_focal_data))
X = np.vstack((focal_data, non_focal_data))
y = np.concatenate((focal_labels, non_focal_labels))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.06, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

selector = SelectKBest(score_func=mutual_info_classif, k=12)
X_train_selected = selector.fit_transform(X_train_balanced, y_train_balanced)
X_test_selected = selector.transform(X_test_scaled)


def evaluate_model_random_search(model, params, model_name, n_iter=4):
    random_search = RandomizedSearchCV(
        model, param_distributions=params, n_iter=n_iter, cv=5, scoring='accuracy', n_jobs=-1, random_state=42
    )
    random_search.fit(X_train_selected, y_train_balanced)
    y_pred = random_search.predict(X_test_selected)
    print(f"{model_name} Best Params:", random_search.best_params_)
    print(f"{model_name} Report:")
    print(classification_report(y_test, y_pred))
    print(f"{model_name} Accuracy:", accuracy_score(y_test, y_pred))


gb_params = {
    'n_estimators': [600], 'learning_rate': [0.2, 0.25], 'max_depth': [13],
    'min_samples_split': [5], 'min_samples_leaf': [10], 'subsample': [1.0],
    'max_features': ['log2']
}
evaluate_model_random_search(GradientBoostingClassifier(), gb_params, 'Gradient Boosting')


xgboost_params = {
    'n_estimators': [800], 'learning_rate': [0.15], 'max_depth': [13],
    'min_child_weight': [4], 'subsample': [1], 'colsample_bytree': [1.0]
}
evaluate_model_random_search(XGBClassifier(eval_metric='logloss'), xgboost_params, 'XGBoost')


In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.metrics import accuracy_score, classification_report

def lstm_model(input_shape):
    model = Sequential()
    
    model.add(LSTM(256, return_sequences=True, input_shape=input_shape, kernel_regularizer=l2(0.001)))
    model.add(BatchNormalization())
    model.add(Dropout(0.4))
    
    model.add(LSTM(128, return_sequences=True, kernel_regularizer=l2(0.001)))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    
    model.add(LSTM(64, return_sequences=False, kernel_regularizer=l2(0.001)))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    
    model.add(Dense(64, activation='relu', kernel_regularizer=l2(0.001)))
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='sigmoid'))  
    
    optimizer = Adam(learning_rate=0.0005, clipvalue=1.0)  
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

X_train_lstm = X_train_balanced.reshape((X_train_balanced.shape[0], 1, X_train_balanced.shape[1]))
X_test_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

final_lstm_model = lstm_model((X_train_lstm.shape[1], X_train_lstm.shape[2]))

history = final_lstm_model.fit(
    X_train_lstm, y_train_balanced,
    epochs=1500, 
    batch_size=128, 
    verbose=2
)

y_pred = (final_lstm_model.predict(X_test_lstm) > 0.5).astype(int)
print("Final LSTM Model Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Epoch 1/1500
56/56 - 2s - 31ms/step - accuracy: 0.6629 - loss: 1.2086
Epoch 2/1500
56/56 - 0s - 4ms/step - accuracy: 0.7410 - loss: 1.0374
Epoch 3/1500
56/56 - 0s - 6ms/step - accuracy: 0.7727 - loss: 0.9517
Epoch 4/1500
56/56 - 0s - 6ms/step - accuracy: 0.7841 - loss: 0.9098
Epoch 5/1500
56/56 - 0s - 6ms/step - accuracy: 0.7899 - loss: 0.8628
Epoch 6/1500
56/56 - 0s - 6ms/step - accuracy: 0.7982 - loss: 0.8331
Epoch 7/1500
56/56 - 0s - 6ms/step - accuracy: 0.8097 - loss: 0.8105
Epoch 8/1500
56/56 - 0s - 6ms/step - accuracy: 0.8115 - loss: 0.7839
Epoch 9/1500
56/56 - 0s - 6ms/step - accuracy: 0.8170 - loss: 0.7497
Epoch 10/1500
56/56 - 0s - 6ms/step - accuracy: 0.8228 - loss: 0.7348
Epoch 11/1500
56/56 - 0s - 6ms/step - accuracy: 0.8259 - loss: 0.7141
Epoch 12/1500
56/56 - 0s - 6ms/step - accuracy: 0.8294 - loss: 0.6834
Epoch 13/1500
56/56 - 0s - 6ms/step - accuracy: 0.8303 - loss: 0.6723
Epoch 14/1500
56/56 - 0s - 6ms/step - accuracy: 0.8413 - loss: 0.6458
Epoch 15/1500
56/56 - 0s - 6